# 06 — Genetic programming per β(t): esperimento sulla seconda ondata

**Prima di questo notebook: il notebook 05** verifica su epidemie sintetiche (β vera nota) che cosa il GP sa ritrovare. Le conclusioni qui sotto valgono solo nei limiti che il notebook 05 mostra (ritardo identificabile? falsi positivi della mobilità?).

**Impostazione** (decisioni dell'11/11/2026):
- finestra **1/9/2020 – 31/1/2021**, training sui primi 2/3 (fino all'11/12/2020), test sul resto;
- **β(t) = β_cost · exp(g(x(t)))**, con g l'albero evoluto e β_cost il β della baseline costante; β limitata a [10⁻⁴, 2];
- **tre configurazioni di terminali**: solo `t` (in mesi); solo mobilità (`retail`, `work` con ritardi 0, 3, 7, 10, 14); entrambi;
- σ fisso (**1/4** di riferimento, **1/3** come sensibilità; la prova pilota era con 1/5,2), γ(t) e μ(t) dai dati, **E₀ stimato per ogni formula** come nelle baseline;
- NSGA-II su (errore normalizzato sul training, numero di nodi), (μ + λ) con 200 + 200 individui, 100 generazioni, crossover oppure mutazione, profondità ≤ 6, duplicati eliminati, costanti del front ottimizzate con Nelder-Mead ogni 5 generazioni;
- **10 seed** per configurazione e per σ.

Codice in `src/seird_gp/gp.py`; run con `scripts/run_gp.py` (circa 7 minuti l'una, 60 run in circa 4 ore con 2 core). Qui si leggono i Pareto front salvati in `results/gp/`.

**Regola per scegliere una formula per run** (fissata prima di guardare i risultati): la formula più semplice del Pareto front con errore di training entro il 5% del migliore di quella run. Usa solo il training: il test resta un controllo onesto. La regola del "gomito" è stata scartata dopo la prova pilota: con front di 3–7 formule sceglieva formule banali. (Lo script `run_gp.py` stampa ancora la formula al gomito: è solo un'anteprima, la scelta si fa qui sui front salvati.)

**Robustezza della regola** (sezione 1b): la soglia del 5% è arbitraria, quindi la scelta si ripete con soglie 0%, 2%, 5%, 10%, 20%. Le conclusioni valgono solo se non cambiano con la soglia.

In [ ]:
import glob, json, pickle, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from seird_gp.data import PROCESSED_DIR, RESULTS_DIR
from seird_gp.beta_models import load_restrictions

TOL = 0.05
SIGMA_KEYS = {"4": "σ = 1/4 (latenza)", "3": "σ = 1/3 (latenza, Europa)", "5.2": "σ = 1/5.2"}
CONFIG_ORDER = ["tempo", "mobilita", "tempo+mobilita"]

def load_runs(tag="full_"):
    rows = []
    for f in sorted(glob.glob(str(RESULTS_DIR / "gp" / f"{tag}*.json"))):
        m = re.search(r"_s([\d.]+)_(.+)_seed(\d+)\.json$", f)
        front = pd.DataFrame(json.load(open(f)))
        front["beta"] = front["beta"].apply(np.array)
        rows.append({"σ": m.group(1), "config": m.group(2), "seed": int(m.group(3)), "front": front})
    return pd.DataFrame(rows)

def choose(front, tol=TOL):
    ok = front[front["fitness_norm train"] <= (1 + tol) * front["fitness_norm train"].min()]
    return ok.sort_values(["nodi", "fitness_norm train"]).iloc[0]

runs = load_runs()
chosen = pd.DataFrame([{**{k: r[k] for k in ["σ", "config", "seed"]}, **choose(r["front"]).to_dict()}
                       for _, r in runs.iterrows()])
print(len(runs), "run;", runs.groupby(["σ", "config"]).size().to_dict())

## 1. Il GP contro le baseline

Mediana e intervallo interquartile sui 10 seed della formula scelta, accanto alle baseline dello stesso σ (notebook 04).

In [ ]:
base = pickle.load(open(RESULTS_DIR / "04_baseline_all.pkl", "rb"))
W = "set20–gen21"
rows = []
for s, key in SIGMA_KEYS.items():
    for m in ["costante", "a tratti", "lineare retail", "lineare work", "lineare retail+work"]:
        if (W, m, key) in base:
            r = base[(W, m, key)]
            rows.append({"σ": s, "modello": f"baseline {m}", "train": r.rmsd.loc["train", "fitness_norm"],
                         "test": r.rmsd.loc["test", "fitness_norm"], "test IQR": "", "nodi": ""})
    for c in CONFIG_ORDER:
        g = chosen[(chosen["σ"] == s) & (chosen["config"] == c)]
        if len(g):
            q = g["fitness_norm test"].quantile([.25, .75])
            rows.append({"σ": s, "modello": f"GP {c} (mediana su {len(g)} seed)",
                         "train": g["fitness_norm train"].median(), "test": g["fitness_norm test"].median(),
                         "test IQR": f"{q.iloc[0]:.2f}–{q.iloc[1]:.2f}", "nodi": int(g["nodi"].median())})
summary = pd.DataFrame(rows).set_index(["σ", "modello"])
summary.to_csv(RESULTS_DIR / "06_gp_vs_baseline.csv")
summary.round(3)

## 1b. Sensibilità sulla soglia di scelta

La formula di ogni run si sceglie con una soglia: la più semplice con errore di training entro il `tol` del migliore della run. Il 5% è arbitrario, quindi qui la scelta si ripete con **tol = 0%, 2%, 5%, 10%, 20%**:
- **0%** = la formula migliore sul training (la più complessa, a rischio di overfitting);
- soglie crescenti = formule sempre più semplici.

Come prima, la scelta usa solo il training; il test serve solo a valutare.

**Criterio, fissato prima di guardare i risultati.** La regola è robusta se, per ogni σ, al variare di `tol`:
1. l'ordine delle tre configurazioni per errore di test mediano non cambia;
2. il confronto di ogni configurazione con le baseline (costante, migliore lineare, a tratti) non cambia: se il GP batte una baseline al 5%, la batte a tutte le soglie (e viceversa).

Se una delle due cose cambia, la conclusione corrispondente va riportata come dipendente dalla regola di scelta.


In [ ]:
TOLS = [0.0, 0.02, 0.05, 0.10, 0.20]

rows = []
for tol in TOLS:
    for _, r in runs.iterrows():
        f = choose(r["front"], tol)
        rows.append({"tol": tol, "σ": r["σ"], "config": r["config"], "seed": r["seed"],
                     "formula": f["formula"], "nodi": int(f["nodi"]),
                     "train": f["fitness_norm train"], "test": f["fitness_norm test"]})
sens = pd.DataFrame(rows)

# quota di run in cui la formula scelta è la stessa della soglia di riferimento (5%)
ref = sens[sens["tol"] == TOL].set_index(["σ", "config", "seed"])["formula"]
sens["come al 5%"] = [f == ref[(s, c, k)] for s, c, k, f in sens[["σ", "config", "seed", "formula"]].itertuples(index=False)]

agg = (sens.groupby(["σ", "config", "tol"])
       .agg(test_mediana=("test", "median"),
            test_q25=("test", lambda x: x.quantile(.25)),
            test_q75=("test", lambda x: x.quantile(.75)),
            nodi_mediana=("nodi", "median"),
            quota_come_5=("come al 5%", "mean"))
       .reset_index())
agg.to_csv(RESULTS_DIR / "06_sensibilita_soglia.csv", index=False)

def wide(col, fmt="{:.3f}"):
    t = agg.pivot_table(index=["σ", "config"], columns="tol", values=col)
    t.columns = [f"{c:.0%}" for c in t.columns]
    return t.reindex(CONFIG_ORDER, level="config")

print("Errore di test, mediana sui seed")
display(wide("test_mediana").round(3))
print("Numero di nodi, mediana sui seed")
display(wide("nodi_mediana"))
print("Quota di run con la stessa formula scelta al 5%")
display(wide("quota_come_5").round(2))


Verifica del criterio: per ogni σ e soglia, l'ordine delle configurazioni e il confronto con le baseline (✓ = GP migliore della baseline sul test, ✗ = peggiore). Le baseline vengono da `results/04_baseline_all.pkl` (sezione 1); se per un σ mancano, la colonna resta vuota.


In [ ]:
LINEARI = ["lineare retail", "lineare work", "lineare retail+work"]

def baseline_test(s):
    key = SIGMA_KEYS[s]
    get = lambda m: base[(W, m, key)].rmsd.loc["test", "fitness_norm"] if (W, m, key) in base else np.nan
    lin = [get(m) for m in LINEARI]
    return {"costante": get("costante"),
            "migliore lineare": np.nanmin(lin) if not np.all(np.isnan(lin)) else np.nan,
            "a tratti": get("a tratti")}

def mark(gp, b):
    return "" if np.isnan(b) else ("✓" if gp < b else "✗")

check = []
for (s, tol), g in agg.groupby(["σ", "tol"]):
    med = g.set_index("config")["test_mediana"].reindex(CONFIG_ORDER).dropna()
    b = baseline_test(s)
    row = {"σ": s, "soglia": f"{tol:.0%}", "ordine (test mediano)": " < ".join(med.sort_values().index)}
    for c in med.index:
        for name, v in b.items():
            row[f"{c} vs {name}"] = mark(med[c], v)
    check.append(row)
check = pd.DataFrame(check).set_index(["σ", "soglia"])

stable = check.groupby(level="σ").nunique().eq(1).T
stable.columns = [f"σ = 1/{s}" for s in stable.columns]
print("Criterio rispettato (True = uguale a tutte le soglie):")
display(stable)
check.T


In [ ]:
sigmas = sorted(agg["σ"].unique(), key=float)
fig, axes = plt.subplots(1, len(sigmas), figsize=(5.5 * len(sigmas), 4), sharey=True, squeeze=False)
colors = {"tempo": "#7f7f7f", "mobilita": "#1f4e79", "tempo+mobilita": "#2a9d8f"}
styles = {"costante": ":", "migliore lineare": "--", "a tratti": "-."}
x = np.array(TOLS) * 100
for ax, s in zip(axes[0], sigmas):
    for c in CONFIG_ORDER:
        g = agg[(agg["σ"] == s) & (agg["config"] == c)].sort_values("tol")
        if len(g):
            ax.plot(x, g["test_mediana"], "o-", color=colors[c], label=f"GP {c}")
            ax.fill_between(x, g["test_q25"], g["test_q75"], color=colors[c], alpha=0.15)
    for name, v in baseline_test(s).items():
        if not np.isnan(v):
            ax.axhline(v, color="#a33", ls=styles[name], lw=1, label=f"baseline {name}")
    ax.axvline(TOL * 100, color="black", lw=0.5)
    ax.set_yscale("log")
    ax.set_xticks(x)
    ax.set_xlabel("soglia (% sopra il miglior errore di training)")
    ax.set_title(SIGMA_KEYS[s])
axes[0][0].set_ylabel("errore di test normalizzato (mediana, IQR)")
axes[0][-1].legend(fontsize=8, loc="upper right")
plt.tight_layout()


**Lettura.** *(da scrivere dopo i risultati: il criterio è rispettato? Se no, quali conclusioni dipendono dalla soglia?)*


## 2. Le formule scelte

Una riga per run. `ritardi` = ritardi della mobilità usati dalla formula.

In [ ]:
def lags(terms):
    return sorted({int(t.split("lag")[1]) for t in terms if "lag" in t})

chosen["ritardi"] = chosen["terminali usati"].apply(lags)
chosen["usa t"] = chosen["terminali usati"].apply(lambda t: "t" in t)
cols = ["σ", "config", "seed", "nodi", "formula", "fitness_norm train", "fitness_norm test", "E0", "ritardi", "usa t"]
chosen.sort_values(["σ", "config", "seed"])[cols].round(3)

## 3. Quali ritardi della mobilità sceglie il GP

Criterio del notebook 03, fissato prima dei risultati: se il GP usa di solito ritardi di 0–3 giorni il modello è coerente; se usa di solito 7 giorni o più manca il ritardo di notifica. Si guarda la distribuzione sui seed, non una singola run.

In [ ]:
mob = chosen[chosen["config"] != "tempo"].copy()
mob["ritardo minimo"] = mob["ritardi"].apply(lambda l: min(l) if l else np.nan)
mob["ritardo massimo"] = mob["ritardi"].apply(lambda l: max(l) if l else np.nan)
counts = (mob.explode("ritardi").dropna(subset=["ritardi"])
          .groupby(["σ", "config", "ritardi"]).size().unstack(fill_value=0))
print("Formule che usano almeno un indice di mobilità:", int(mob["ritardi"].apply(bool).sum()), "su", len(mob))
print("Quota con tutti i ritardi ≤ 3 giorni:", round((mob["ritardo massimo"] <= 3).mean(), 2))
print("Quota con almeno un ritardo ≥ 7 giorni:", round((mob["ritardo massimo"] >= 7).mean(), 2))
counts

## 4. β(t) delle formule scelte

In [ ]:
data = pd.read_csv(PROCESSED_DIR / "trento_clean.csv", index_col=0, parse_dates=True)
idx = data.loc["2020-09-01":"2021-01-31"].index
restr = load_restrictions()
s_ref = "4" if (chosen["σ"] == "4").any() else chosen["σ"].iloc[0]
fig, ax = plt.subplots(len(CONFIG_ORDER), 1, figsize=(11, 9), sharex=True, sharey=True)
pw = base.get((W, "a tratti", SIGMA_KEYS[s_ref]))
for a, c in zip(ax, CONFIG_ORDER):
    for _, r in chosen[(chosen["σ"] == s_ref) & (chosen["config"] == c)].iterrows():
        a.plot(idx, r["beta"], color="#1f4e79", alpha=0.35, lw=1)
    if pw is not None:
        a.plot(pw.extra["beta"], color="#a33", lw=1.5, label="baseline a tratti")
    a.axvspan(idx[int(round(len(idx) * 2 / 3))], idx[-1], color="#f4e3c1", alpha=0.6)
    for d in restr.inizio[(restr.inizio > idx[0]) & (restr.inizio <= idx[-1])]:
        a.axvline(d, color="#a33", lw=0.5, ls="--")
    a.set_title(f"GP {c}, σ = 1/{s_ref}: formule scelte dei 10 seed (blu)", fontsize=9)
    a.set_ylabel("β(t)")
    a.spines[["top", "right"]].set_visible(False)
ax[0].legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(RESULTS_DIR / "05_beta_gp.png", dpi=150)

## 5. Lettura

*(da scrivere dopo i risultati)*